In [1]:
import numpy as np
import pandas as pd
import torch
from rouge_score import rouge_scorer 
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm.notebook import tqdm

In [2]:
DATA_DIR = Path(".")
train = pd.read_csv(DATA_DIR/"Train.csv")
val   = pd.read_csv(DATA_DIR/"Val.csv")
test  = pd.read_csv(DATA_DIR/"Test.csv")

QCOL, ACOL, GCOL, IDCOL = "input", "output", "subset", "ID"

for df, name in [(train,"train"),(val,"val"),(test,"test")]:
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna("").astype(str).str.strip()
train = train[(train[QCOL]!="")&(train[ACOL]!="")].reset_index(drop=True)
val   = val[(val[QCOL]!="")&(val[ACOL]!="")].reset_index(drop=True)
print(f"train {len(train)}  val {len(val)}  test {len(test)}")
print(train[GCOL].value_counts())

train 29814  val 6686  test 2618
subset
Eng_Uga    7623
Aka_Gha    4455
Eng_Gha    4443
Eng_Eth    3915
Lug_Uga    3383
Eng_Ken    2080
Swa_Ken    2070
Amh_Eth    1845
Name: count, dtype: int64


In [3]:
train.head()

,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nn...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu...,"Wɔ Ghana mu no, amanmmra no gye binary gender ...",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔ...,Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔ...,Aka_Gha
3,ID_TR_Aka_Gha_BDD640FB,"Dɛn ne aduru a wodi si nyisɛn ano ntɛm ntɛm, n...",Aduru a wodi si nyisɛn ano ntɛm ntɛm (Emergen...,Aka_Gha
4,ID_TR_Aka_Gha_46685257,Hu sɛnea ɛyɛ den sɛ wubehu bɔbea mu basabasayɔ...,Ɛnyɛ mmerɛw bere nyinaa sɛ wubehu bɔbea mu bas...,Aka_Gha


In [4]:
class WhitespaceTokenizer:
    def tokenize(self, text):
        if text is None: return []
        return str(text).strip().split()

_SCORER = rouge_scorer.RougeScorer(["rouge1","rougeL"],
                                   tokenizer=WhitespaceTokenizer(), use_stemmer=False)

def compute_rouge(preds, refs):
    r1, rl = [], []
    for p, r in zip(preds, refs):
        s = _SCORER.score(str(r), str(p))
        r1.append(s["rouge1"].fmeasure); rl.append(s["rougeL"].fmeasure)
    return {"rouge1_f1": float(np.mean(r1)) if r1 else 0.0,
            "rougeL_f1": float(np.mean(rl)) if rl else 0.0}

def compute_single_rouge(pred,ref):
    s = _SCORER.score(pred,ref)
    return s["rouge1"].fmeasure, s["rougeL"].fmeasure

def rouge_by_subset(preds, refs, subs):
    sub = np.array(subs); rows=[]
    for s in np.unique(sub):
        m = sub==s
        rows.append({"subset":s, "n":int(m.sum()),
                     **compute_rouge([p for p,k in zip(preds,m) if k],
                                     [x for x,k in zip(refs,m) if k])})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"subset":"OVERALL(micro)","n":len(preds),
                       **compute_rouge(preds, refs)}
    return df

In [5]:
sub = "Amh_Eth"

Train_Gha = train[train['subset'] == sub]
Val_Gha = val[val['subset'] == sub]

r1s = []
rLs = []
for idx,val_row in tqdm(Val_Gha.iterrows(),total=len(Val_Gha)):
    
    candidate_r1s = []
    candidate_rLs = []

    ref = val_row['output']

    for tr_idx,train_row in Train_Gha.iterrows():
        
        pred = train_row['output']
        r1,rL = compute_single_rouge(pred,ref)
        candidate_r1s.append(r1)
        candidate_rLs.append(rL)

    r1s.append(max(candidate_r1s))
    rLs.append(max(candidate_rLs))

    print(f"Running ROUGE1 mean = {sum(r1s)/len(r1s)}")


  0%|          | 0/462 [00:00<?, ?it/s]

Running ROUGE1 mean = 0.2272727272727273
Running ROUGE1 mean = 0.23558758314855877
Running ROUGE1 mean = 0.23398146568878278
Running ROUGE1 mean = 0.2588194325999204
Running ROUGE1 mean = 0.2670555460799363
Running ROUGE1 mean = 0.31556954421390043
Running ROUGE1 mean = 0.35479731424423316
Running ROUGE1 mean = 0.37583226534831937
Running ROUGE1 mean = 0.37111016179109874
Running ROUGE1 mean = 0.36158535250854057
Running ROUGE1 mean = 0.3765608467780991
Running ROUGE1 mean = 0.3600617285942099
Running ROUGE1 mean = 0.3622792024630313
Running ROUGE1 mean = 0.3518461320168303
Running ROUGE1 mean = 0.3765378713638564
Running ROUGE1 mean = 0.3934454308742036
Running ROUGE1 mean = 0.38302018295314555
Running ROUGE1 mean = 0.3778703161582575
Running ROUGE1 mean = 0.37692977320255977
Running ROUGE1 mean = 0.39808328454243175
Running ROUGE1 mean = 0.3886507471832683
Running ROUGE1 mean = 0.3853388711151293
Running ROUGE1 mean = 0.3779017773399374
Running ROUGE1 mean = 0.36973162752653094
Runni